In [1]:
import xarray as xr
import rasterio
from shapely.geometry import Polygon
import geopandas as gpd
import numpy as np
import os
import pandas as pd
from utils.data_prep import h3_grid as h3
from utils.ahp import Ahp_calc
from utils.geo_score_converter import GeoIntervalScorer
%load_ext autoreload
%autoreload 2

In [4]:
# wind_speed
path = 'COL_wind-speed_10m.tif'
ds_wind = rasterio.open(path)
crs = 'EPSG:9377'
resolution = 8

h3_module = h3(
    raster=ds_wind,
    resolution=8,
    crs=crs
)

path = 'seeds/codigo_tipos(in).csv'
df_data = pd.read_csv(path,sep=';',encoding='latin-1')
df_data = df_data.drop(['Criterio'], axis = 1)
df_data = df_data[~df_data['Subcriterio'].isna()]

In [2]:
codigos_miro = [
    "AMB-003-A",
    "AMB-005-A", # no tiene crs 4326
    "AMB-006-A",
    "AMB-007-AT",
    "AMB-008-AT",
    "AMB-012-A",
    "AMB-017-A",
    "AMB-019-A",
    "AMB-020-A",
    "AMB-021-A",
    "AMB-028-AT",
    "AMB-029-AT",
    "AMB-030-AT",
    "AMB-031-A",
    "AMB-033-A",   
    "AMB-034-A",
    "AMB-036-A",
    "AMB-038-A",
    "AMB-040-A",
    "AMB-042-A"
]
len(codigos_miro)

20

In [5]:
df = df_data[df_data['Codigo'].isin(codigos_miro)].reset_index(drop=True)
df

,Codigo,Variable,Tipo,Subcriterio,Invertir
0,AMB-003-A,Especies migratorias,binario,Especies,si
1,AMB-005-A,Especies endï¿½micas,continua,Especies,si
2,AMB-006-A,Cobertura de suelos,categorica,Condiciones,no
3,AMB-007-AT,Pendiente del terreno,continua,Condiciones,no
4,AMB-008-AT,Elevaciï¿½n,continua,Condiciones,no
5,AMB-012-A,Humedales RAMSAR,binario,Areas,si
6,AMB-017-A,Especies aves IUCN,continua,Especies,si
7,AMB-019-A,Especies marinas listadas en la IUCN,binario,Especies,si
8,AMB-020-A,Pastos marinos,binario,Especies,si
9,AMB-021-A,Areas Coralinas,binario,Areas,si


In [7]:
map = gpd.read_file('preliminary-map')

In [ ]:
convertir = ['AMB-003-A','AMB-006-A','AMB-007-AT','AMB-008-AT','AMB-029-AT','AMB-030-AT','AMB-012-A','AMB-019-A','AMB-020-A','AMB-021-A','AMB-031-A','AMB-033-A','AMB-034-A','AMB-036-A','AMB-038-A']
trans = GeoIntervalScorer('seeds/intervalos.csv','seeds/codigo_tipos(in).csv')
temp = trans.transform(map, columns=convertir)

In [ ]:

for col in temp.columns:
    try:
        col_min = temp[col].min()
        col_max = temp[col].max()
        print(f"Columna: {col} | Mínimo: {col_min} | Máximo: {col_max}")
    except Exception as e:
        print(f"No se pudo calcular min/max para la columna {col}: {e}")

Columna: h3 | Mínimo: 88e628a401fffff | Máximo: 88f16bdb6dfffff
Columna: AMB-040-A | Mínimo: 0.0 | Máximo: 5.0
Columna: AMB-003-A | Mínimo: 0 | Máximo: 5
Columna: AMB-005-A | Mínimo: 0.555555555555556 | Máximo: 5.0
Columna: AMB-006-A | Mínimo: 0 | Máximo: 5
Columna: AMB-007-AT | Mínimo: 0.0 | Máximo: 5.0
Columna: AMB-008-AT | Mínimo: 0.0 | Máximo: 5.0
Columna: AMB-012-A | Mínimo: 0.0 | Máximo: 5.0
Columna: AMB-017-A | Mínimo: 0.234249758508012 | Máximo: 5.0
Columna: AMB-019-A | Mínimo: 0 | Máximo: 5
Columna: AMB-020-A | Mínimo: 0 | Máximo: 5
Columna: AMB-021-A | Mínimo: 0 | Máximo: 5
Columna: AMB-028-AT | Mínimo: 1.0 | Máximo: 5.0
Columna: AMB-029-AT | Mínimo: 0 | Máximo: 5
Columna: AMB-030-AT | Mínimo: 0.0 | Máximo: 5.0
Columna: AMB-031-A | Mínimo: 0 | Máximo: 5
Columna: AMB-033-A | Mínimo: 0 | Máximo: 5
Columna: AMB-034-A | Mínimo: 0 | Máximo: 5
Columna: AMB-036-A | Mínimo: 0 | Máximo: 5
Columna: AMB-038-A | Mínimo: 0 | Máximo: 5
Columna: AMB-042-A | Mínimo: 0.234249758508012 | Máxim

In [9]:
# # Reemplaza los valores de var en map por los valores de xr,
# # asegurando que el tipo de columna de map[var] soporte los nuevos valores (posiblemente floats)
# var = 'AMB-017-A'
# temp = df_data[df_data['Codigo'].isin([f'{var}'])].reset_index(drop=True)
# xr = h3_module.vars_ahp('vars_ambiental_ahp/', temp)

# # Nos aseguramos de que ambos DataFrames tengan la clave de unión 'h3'
# if  var in xr.columns:
#     # map = map.set_index('h3')
#     # xr = xr.set_index('h3')
#     # Convertimos la columna objetivo en float para prevenir errores de asignación de tipo
#     if var in map.columns:
#         map[var] = map[var].astype(float)
#     else:
#         # Si la columna no existe, crea una columna vacía de tipo float
#         map[var] = np.nan
#     # Actualiza los valores solo en los índices presentes en ambos DataFrames
#     map.loc[xr.index, var] = xr[var]
#     map = map.reset_index()
# else:
#     print("No se encontraron las columnas necesarias para reemplazo.")

In [9]:
temp = temp.fillna(0)

In [10]:
from utils.pesos_variables import pesos_dict

%load_ext autoreload
%autoreload 2

SUBCRITERIO_A_CATEGORIA = {
    "Especies": "Especies Bióticas",
    "Areas": "Áreas Protegidas",
    "Condiciones": "Geología/Geotecnia",
    "Amenazas": "Amenazas",
}

codigo_subcriterio = (
    df_data.set_index("Codigo")["Subcriterio"].astype(str).str.strip().to_dict()
)
pesos_local = pesos_dict()

amb_cols = [c for c in temp.columns if c.startswith("AMB-")]

cols_por_subcriterio = {}
for codigo in amb_cols:
    sub = codigo_subcriterio.get(codigo)
    if sub and sub in SUBCRITERIO_A_CATEGORIA:
        cols_por_subcriterio.setdefault(sub, []).append(codigo)

scores_subcriterio = pd.DataFrame(index=temp.index)

for sub, cols in cols_por_subcriterio.items():
    cols_con_peso = [c for c in cols if c in pesos_local]
    cols_sin_peso = [c for c in cols if c not in pesos_local]
    if cols_sin_peso:
        print(f"Sin peso AHP en '{sub}': {cols_sin_peso}")
    if not cols_con_peso:
        continue

    pesos = np.array([pesos_local[c] for c in cols_con_peso])
    pesos = pesos / pesos.sum()

    scores_subcriterio[sub] = temp[cols_con_peso].mul(pesos, axis=1).sum(axis=1)

resultado = temp.copy()
for sub in scores_subcriterio.columns:
    resultado[sub] = scores_subcriterio[sub]

print("Pesos locales AHP:")
for codigo, peso in pesos_local.items():
    print(f"  {codigo}: {peso:.4f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Pesos locales AHP:
  AMB-005-A: 0.2150
  AMB-003-A: 0.0469
  AMB-019-A: 0.2150
  AMB-017-A: 0.2150
  AMB-042-A: 0.2150
  AMB-038-A: 0.0258
  AMB-034-A: 0.0673
  AMB-033-A: 0.0743
  AMB-012-A: 0.2412
  AMB-020-A: 0.1353
  AMB-021-A: 0.4137
  AMB-036-A: 0.1353
  AMB-040-A: 0.7500
  AMB-009-A: 0.2500
  AMB-031-A: 0.4059
  AMB-028-AT: 0.2129
  AMB-030-AT: 0.1456
  AMB-029-AT: 0.0978
  AMB-006-A: 0.0649
  AMB-007-AT: 0.0431
  AMB-008-AT: 0.0298


In [16]:
pesos_local

{'AMB-005-A': np.float64(0.21499997789695538),
 'AMB-003-A': np.float64(0.04693471500706826),
 'AMB-019-A': np.float64(0.21499997789695538),
 'AMB-017-A': np.float64(0.21499997789695538),
 'AMB-042-A': np.float64(0.21499997789695538),
 'AMB-038-A': np.float64(0.025794970767605955),
 'AMB-034-A': np.float64(0.06727040263750415),
 'AMB-033-A': np.float64(0.07433730588660166),
 'AMB-012-A': np.float64(0.24122788010111954),
 'AMB-020-A': np.float64(0.1353485012639942),
 'AMB-021-A': np.float64(0.41373781148429034),
 'AMB-036-A': np.float64(0.1353485012639942),
 'AMB-040-A': np.float64(0.75),
 'AMB-009-A': np.float64(0.25),
 'AMB-031-A': np.float64(0.40592052789548555),
 'AMB-028-AT': np.float64(0.21289265026222598),
 'AMB-030-AT': np.float64(0.14558793092206737),
 'AMB-029-AT': np.float64(0.09779089369974188),
 'AMB-006-A': np.float64(0.06491186219921725),
 'AMB-007-AT': np.float64(0.04314138734717481),
 'AMB-008-AT': np.float64(0.029754747674087208)}

In [15]:
cols_por_subcriterio

{'Amenazas': ['AMB-040-A'],
 'Especies': ['AMB-003-A',
  'AMB-005-A',
  'AMB-017-A',
  'AMB-019-A',
  'AMB-020-A',
  'AMB-034-A',
  'AMB-038-A',
  'AMB-042-A'],
 'Condiciones': ['AMB-006-A',
  'AMB-007-AT',
  'AMB-008-AT',
  'AMB-028-AT',
  'AMB-029-AT',
  'AMB-030-AT',
  'AMB-031-A'],
 'Areas': ['AMB-012-A', 'AMB-021-A', 'AMB-033-A', 'AMB-036-A']}

In [11]:
resultado = resultado[["h3", "Amenazas", "Especies", "Condiciones", "Areas","geometry"]]

In [12]:
from utils.pesos_variables import (
    PESO_RAMA_ABIOTICAS,
    PESO_RAMA_BIOTICAS,
    combinar_score_ahp_global,
    combinar_score_grupo,
    pesos_grupo_subcriterio,
)

subcriterios = ["Amenazas", "Especies", "Condiciones", "Areas"]

resultado["score_bioticas"] = combinar_score_grupo(resultado, "bioticas")
resultado["score_abioticas"] = combinar_score_grupo(resultado, "abioticas")
resultado["score_ahp"] = combinar_score_ahp_global(resultado[subcriterios])

print("Pesos dentro de bióticas (%):")
for sub, peso in pesos_grupo_subcriterio("bioticas").items():
    print(f"  {sub}: {peso * 100:.1f}%")

print("\nPesos dentro de abióticas (%):")
for sub, peso in pesos_grupo_subcriterio("abioticas").items():
    print(f"  {sub}: {peso * 100:.1f}%")

print(f"\nPeso rama biótica: {PESO_RAMA_BIOTICAS * 100:.0f}%")
print(f"Peso rama abiótica: {PESO_RAMA_ABIOTICAS * 100:.0f}%")

resultado[[
    "h3", "Especies", "Areas", "score_bioticas",
    "Condiciones", "Amenazas", "score_abioticas", "score_ahp","geometry"
]].head()

Pesos dentro de bióticas (%):
  Especies: 25.0%
  Areas: 75.0%

Pesos dentro de abióticas (%):
  Condiciones: 66.7%
  Amenazas: 33.3%

Peso rama biótica: 50%
Peso rama abiótica: 50%


,h3,Especies,Areas,score_bioticas,Condiciones,Amenazas,score_abioticas,score_ahp,geometry
0,88e74ab153fffff,4.590407,5.0,4.897602,3.594563,4.200,3.796375,4.346989,"POLYGON ((5325672.764 2270751.835, 5326192.707..."
1,88e74e391bfffff,5.000000,5.0,5.000000,3.122023,0.000,2.081349,3.540674,"POLYGON ((5277632.122 2453118.543, 5278155.352..."
2,88ef9e3307fffff,4.793303,5.0,4.948326,3.305161,1.825,2.811774,3.88005,"POLYGON ((4714660.425 1703928.869, 4715184.234..."
3,88e75c583bfffff,4.053154,5.0,4.763289,2.029603,0.000,1.353068,3.058178,"POLYGON ((5400394.434 3191045.991, 5400910.973..."
4,88f1693865fffff,4.053154,5.0,4.763289,2.029603,0.000,1.353068,3.058178,"POLYGON ((3845240.623 1229251.959, 3845730.612..."


In [45]:
resultado.to_file('scores_subcriterio_ambiental')

/tmp/ipykernel_94333/1793583583.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  resultado.to_file('scores_subcriterio_ambiental')
/home/choclo/documents/ahpAmbiental/env/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Condiciones' to 'Condicione'
  ogr_write(
/home/choclo/documents/ahpAmbiental/env/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'score_bioticas' to 'score_biot'
  ogr_write(
/home/choclo/documents/ahpAmbiental/env/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'score_abioticas' to 'score_abio'
  ogr_write(
